# Real hardware: ibm_kingston

This notebook sends a batch of labeled boards to a real IBM device in **one
job** (8 readout-calibration circuits, one board with a known winner, and the
boards), saves every result to `results/`, and analyzes the latest saved job.

- **Re-running is free:** by default the notebook only reads saved results.
  Submitting happens only with `RUN_HARDWARE = True`.
- **Only real hardware:** every number here comes from a job that ran on the
  device. Until one has been saved, the notebook says so and the analysis
  cells show nothing; simulated runs are in `02_noise`.

To submit: put `QISKIT_IBM_TOKEN=...` in the repo's `.env` (see `.env.example`),
set `RUN_HARDWARE = True`, run the notebook once, then set it back to False.
The same batch can be sent from a terminal with
`python scripts/collect.py --backend ibm_kingston --n 9 --seed 4903 --tag kingston-1`.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from qttt.backends import get_backend
from qttt.experiments import collect, sample_boards
from qttt.noise import NoiseProfile
from qttt.results import LABELS, RunStore

DEVICE = "ibm_kingston"  # on IBM's free Open plan
RUN_HARDWARE = False  # True submits one job to DEVICE (uses your plan's time)
N_BOARDS = 9  # balanced: 3 no win, 3 single win, 3 double win
SEED = 4903
SHOTS = 1024
TAG = "kingston-1"
STORE = RunStore("../results")

## Submit (only when `RUN_HARDWARE = True`)

In [2]:
boards = sample_boards(N_BOARDS, seed=SEED)
for b in boards:
    print(f"{''.join(c if c != ' ' else '.' for c in b.board)}  {b.player} just moved  {b.label:10} {sorted(b.lines)}")

if RUN_HARDWARE:
    device = get_backend(DEVICE, seed=SEED, transpile_seeds=5)
    print(f"\nSubmitting {len(boards)} boards + 9 calibration circuits to {DEVICE} ({SHOTS} shots each)...")
    collect(boards, device, shots=SHOTS, store=STORE, tag=TAG)
    print("Done; saved to", STORE.runs_file)

XX.OOXO.X  X just moved  no_win     []
XXXOO....  X just moved  single_win [0]
OOXXXXOOX  X just moved  double_win [1, 5]
OOXXO..X.  O just moved  no_win     []
X.OOO.XXX  X just moved  single_win [2]
.X.OOO.XX  O just moved  single_win [1]
OXOXXXOXO  X just moved  double_win [1, 4]
XOOXXXOOX  X just moved  double_win [1, 6]
OX....XXO  X just moved  no_win     []


## Load the latest job on the device

Every analysis below reads from the saved records, so re-running it is free
and needs no IBM account.

In [3]:
def latest_job(store, device):
    """Detection and calibration records of the newest real-hardware job on device."""
    runs = [r for r in store.load("detection") if r["backend"]["name"] == device and not r["backend"]["simulated"]]
    if not runs:
        return None, []
    job = runs[-1]["job_id"]
    detections = [r for r in runs if r["job_id"] == job]
    calibration = next((r for r in store.load("calibration") if r["id"] == detections[0]["calibration_id"]), None)
    return calibration, detections


calibration, detections = latest_job(STORE, DEVICE)
source = STORE
if not detections:
    print(f"No {DEVICE} results saved yet: set RUN_HARDWARE = True above to submit a job.")
else:
    first = detections[0]
    print(f"backend {first['backend']['name']}, job {first['job_id']}, run {first['created'][:19]} UTC")
    print(f"device calibration data from {first['backend']['calibration_date']}, {first['shots']} shots per circuit,"
          f" {len(detections)} boards, tag {first['tag']}")
    print(f"circuit: {first['metrics']['two_qubit_gates']} two-qubit gates, depth {first['metrics']['depth']},"
          f" line qubits {first['metrics']['line_qubits']}")

No ibm_kingston results saved yet: set RUN_HARDWARE = True above to submit a job.


## Noise measured in the job

The 8 readout circuits give the readout matrix A (how often each prepared
line index is read as each other one) and the known-winner board gives ε, the
share of shots gate noise scatters evenly. They set each line's cutoff.

In [4]:
if detections:
    p = calibration["profile"]
    measured = NoiseProfile(epsilon=p["epsilon"], readout=tuple(map(tuple, p["readout"])), source=p["source"],
                            line_qubits=tuple(p["line_qubits"]))
    print(f"ε = {measured.epsilon:.3f}")
    print("correct readout per line index:", [f"{measured.readout[i][i]:.3f}" for i in range(8)])
    print(f"cutoffs at {first['shots']} shots: {first['cutoffs']}")
    needed = measured.shots_needed(0.01, first["budget"])
    print(f"shots needed for ≤1% misses with this noise: {needed or 'more than 16384'}")

    fig, ax = plt.subplots(figsize=(3.6, 3.2))
    im = ax.imshow(measured.readout, cmap="viridis", vmin=0, vmax=1)
    ax.set_xlabel("prepared line index")
    ax.set_ylabel("measured line index")
    ax.set_title("Readout matrix A", fontsize=10)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    plt.show()

## Every board

**Reading the bar charts:** each bar is the fraction of shots measured on a
line index. Green: the board's real winning lines. Gray: losing lines.
Orange: a losing line that reached its cutoff, so it was flagged and then
rejected by the classical check. The black tick on each bar is that line's
cutoff (from the noise profile measured in the job), and the dotted
line is 1/8, an even spread over all 8 lines.

In [5]:
if detections:
    print(f"{'board':10} {'label':10} {'true':6} {'confirmed':9} {'rejected':8}  {'on true lines':>13}  verdict")
    for r in detections:
        true, found = set(r["true_lines"]), set(r["lines"])
        share = sum(r["counts"][i] for i in true) / r["shots"] if true else max(r["counts"]) / r["shots"]
        if not true:
            verdict = "correct" if not found else "WRONG"
        else:
            verdict = "correct" if found == true else "partly missed" if found else "MISSED"
        print(f"{r['board'].replace(' ', '.'):10} {r['label']:10} {str(sorted(true)):6} {str(sorted(found)):9}"
              f" {str(r['rejected']):8}  {share:>13.1%}  {verdict}")

    cols = 3
    rows = -(-len(detections) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.4 * cols, 2.6 * rows), sharey=True, squeeze=False)
    for ax, r in zip(axes.flat, detections):
        true = set(r["true_lines"])
        colors = ["tab:green" if i in true else "tab:orange" if i in r["rejected"] else "tab:gray" for i in range(8)]
        ax.bar(range(8), [c / r["shots"] for c in r["counts"]], color=colors)
        ax.hlines([k / r["shots"] for k in r["cutoffs"]], [i - 0.4 for i in range(8)], [i + 0.4 for i in range(8)],
                  color="black", linewidth=1.5)
        ax.axhline(1 / 8, color="gray", linestyle=":", linewidth=1)
        ax.set_title(f"{r['board'].replace(' ', '.')}  {r['label']}\nfound {r['lines'] or '-'}", fontsize=8)
        ax.set_xticks(range(8))
        ax.set_ylim(0, 1)
    for ax in axes.flat[len(detections):]:
        ax.axis("off")
    fig.suptitle(f"{first['backend']['name']}, {first['shots']} shots")
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    fig.legend(handles=[Patch(color="tab:green", label="winning line"), Patch(color="tab:gray", label="losing line"),
                        Patch(color="tab:orange", label="flagged, rejected by the check"),
                        Line2D([], [], color="black", linewidth=1.5, label="cutoff"),
                        Line2D([], [], color="gray", linestyle=":", label="1/8 (even spread)")],
               loc="lower center", ncol=5, fontsize=8, frameon=False)
    plt.show()

## Summary

A missed win is a board with a winner where no line reached its cutoff. A
false flag is a losing line that reached its cutoff; the classical check
rejects it, so it costs one extra check but never produces a wrong winner.

The budget allows about 0.1 false flags per board. If no-win boards get clearly
more, the noise is not spreading shots evenly over the losing lines as the model
assumes: some lines pick up extra shots even on boards without a winner.

Checks per board counts the classical line checks; checking every line would be
8. That is not a speedup: every shot of the circuit evaluates the oracle on all
8 lines, and each board takes 1024 shots.

In [6]:
if detections:
    print(f"{'label':11} {'boards':>6} {'win found':>10} {'all lines':>10} {'false flags':>11} {'checks/board':>12}")
    for label in LABELS:
        group = [r for r in detections if r["label"] == label]
        if not group:
            continue
        found = sum(bool(r["lines"]) for r in group)
        every = sum(sorted(r["lines"]) == r["true_lines"] for r in group)
        flags = sum(len(r["rejected"]) for r in group)
        checks = sum(len(r["candidates"]) for r in group) / len(group)
        win = label != "no_win"
        print(f"{label:11} {len(group):>6} {f'{found}/{len(group)}' if win else '-':>10}"
              f" {f'{every}/{len(group)}' if win else '-':>10} {flags:>11} {checks:>12.2f}")
    assert all(set(r["lines"]) <= set(r["true_lines"]) for r in detections), "a reported line was not a real win"

## Measured vs predicted

For each board with a winner: the share of shots on its winning line(s),
measured in this job, next to the prediction of the device's published
calibration data (saved with the job when it was submitted, for the same
compiled circuit) and of the noise profile measured in the job itself.

In [7]:
if detections:
    job = next(r for r in source.load("job") if r["id"] == first["job_record"])
    published = NoiseProfile.from_record(job["published_profile"])
    wins = [r for r in detections if r["true_lines"]]
    x = np.arange(len(wins))
    measured_share = [sum(r["counts"][i] for i in r["true_lines"]) / r["shots"] for r in wins]
    model = {"published data": published, "measured profile": measured}
    fig, ax = plt.subplots(figsize=(1.0 * len(wins) + 2.5, 3))
    ax.bar(x - 0.27, measured_share, 0.27, label="hardware result", color="tab:green")
    for k, (name, profile) in enumerate(model.items()):
        predicted = [sum(profile.predicted_for(r["true_lines"])[i] for i in r["true_lines"]) for r in wins]
        ax.bar(x + 0.27 * k, predicted, 0.27, label=name, alpha=0.7)
    ax.axhline(1 / 8, color="gray", linestyle=":", linewidth=1)
    ax.set_xticks(x, [f"{r['board'].replace(' ', '.')}\n{r['label']}" for r in wins], fontsize=7)
    ax.set_ylabel("share of shots on winning lines")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=7)
    fig.tight_layout()
    plt.show()

Both predictions use the same simple model: the shots noise moves are spread
evenly over the 8 lines. The profile measured in the job is fitted on one
known board (winning line 0), so it matches that kind of board best. Gaps on
particular boards show where the model is too simple, for example when the
signal depends on which line won or on which board qubits are set.